# OCR PDF tiếng Việt trên Kaggle — LEGACY

Chạy repo [VuTuanAn1403/OCR_pdf](https://github.com/VuTuanAn1403/OCR_pdf), xuất
Markdown, JSON, `count_keywords.xlsx` cho từng PDF và Excel ma trận tổng hợp
`MA_TRAN_DEM_TU_KHOA_<N>_FILES.xlsx` cho toàn bộ N tài liệu trong input.

1. Trong Kaggle, chọn **File → Import Notebook** để nhập file này.
2. Bật **Internet** và chọn GPU trong phần Settings nếu dùng `DEVICE = "cuda"`.
3. Thêm dataset chứa PDF bằng **Add Input**. File nguồn được đọc từ `/kaggle/input`.
4. Chạy các ô theo thứ tự: thử trang 1 trước, sau đó chạy toàn bộ PDF trong input và xuất ma trận.
5. Đặt `RUN_FULL_BATCH = False` nếu chỉ muốn thử một vài trang; khi đó chưa xuất ma trận cả lô.

`PROFILE = "local"` mặc định dùng VietOCR, có thể chạy GPU hoặc CPU.
Chọn `balanced`/`accuracy` nếu cần thêm Docling cho bảng/công thức; cài đặt và
lần nạp model đầu tiên sẽ lâu hơn. `fast` chỉ dùng CPU. Notebook dùng CLI chuẩn
của dự án; không tự chuyển sang scheduler `unified_gpu` V6.2.

Kết quả và log lưu ở `/kaggle/working/ocr_pdf_results`, file tải về là
`/kaggle/working/ocr_pdf_results.zip`. Môi trường, model và cache đặt trong `/tmp`
để không đưa vào bộ kết quả. Python 3.11 riêng giúp tránh xung đột với kernel Kaggle.

**Kiểm chứng:** file được xác thực cấu trúc/cú pháp và kiểm thử luồng CPU cục bộ.
Chưa thực thi trực tiếp trong phiên Kaggle của bạn; cần Internet, dataset PDF và
GPU/driver tương thích khi chạy chế độ CUDA.

**Pipeline V6.3 GPU:** dùng file [ocr_pdf_v63_kaggle.ipynb](https://github.com/VuTuanAn1403/OCR_pdf/blob/main/kaggle/ocr_pdf_v63_kaggle.ipynb).
Clone mã nguồn mới không thay đổi các cell của notebook đang mở; hãy import lại đúng file notebook.


## 1. Chọn cấu hình và dữ liệu

In [ ]:
from pathlib import Path
import collections
import hashlib
import json
import os
import shutil
import signal
import subprocess
import sys
import time

if sys.platform == "win32":
    for stream in (sys.stdout, sys.stderr):
        if hasattr(stream, "reconfigure"):
            stream.reconfigure(encoding="utf-8", errors="replace")

REPO_URL = "https://github.com/VuTuanAn1403/OCR_pdf.git"
REPO_REF = "main"
PROFILE = "local"                  # local | balanced | accuracy | fast
DEVICE = "cuda"                    # cuda | cpu; fast yêu cầu cpu
INPUT_ROOT = Path("/kaggle/input")  # Có thể đổi thành /kaggle/input/<dataset>
PDF_FILES = []                      # []: tự tìm PDF; hoặc ["dataset/bao_cao.pdf"]
FILE_LIMIT = None                   # Tất cả PDF trong INPUT_ROOT, không giới hạn số file
KEYWORDS_FILE = None                # None: config/keywords.txt; hoặc đường dẫn .txt
SMOKE_PAGES = "1"                   # Ví dụ "1-3"; chỉ dùng cho lượt kiểm tra
RUN_FULL_BATCH = True               # Chạy đủ tài liệu để xuất ma trận tổng hợp
RESUME = True
ASSETS = "jpeg"                     # none | jpeg | png

WORKING_ROOT = Path("/kaggle/working")
RUNTIME_ROOT = Path("/tmp/ocr_pdf_kaggle")
RESULTS_DIR = WORKING_ROOT / "ocr_pdf_results"
LOG_DIR = RESULTS_DIR / "logs"
SMOKE_DIR = RESULTS_DIR / "smoke"
BATCH_DIR = RESULTS_DIR / "documents"
ARCHIVE_PATH = WORKING_ROOT / "ocr_pdf_results.zip"

# Giữ đủ 88 cột đúng thứ tự, bao gồm hai cột Internet di động.
MATRIX_KEYWORDS = """
Trí tuệ nhân tạo
trí tuệ kinh doanh
hiểu biết hình ảnh
hệ thống hỗ trợ quyết định đầu tư
phân tích dữ liệu thông minh
robot thông minh
học máy
học sâu
tìm kiếm ngữ nghĩa
sinh trắc học
nhận dạng khuôn mặt
nhận dạng giọng nói
xác minh danh tính
lái xe tự động
xử lý ngôn ngữ tự nhiên
dữ liệu lớn
khai thác dữ liệu
khai thác văn bản
trực quan hóa dữ liệu
dữ liệu không đồng nhất
báo cáo tín dụng
thực tế tăng cường
thực tế hỗn hợp
thực tế ảo
điện toán đám mây
điện toán dòng
điện toán đồ thị
điện toán bộ nhớ
điện toán bảo mật đa bên
điện toán giống não
điện toán xanh
điện toán nhận thức
kiến trúc hợp nhất
đồng thời cấp tỷ
lưu trữ cấp EB
Internet vạn vật
hệ thống vật lý thông tin
blockchain
tiền tệ số
điện toán phân tán
công nghệ bảo mật riêng tư vi phân
hợp đồng tài chính thông minh
Internet di động
Internet công nghiệp
Internet di động
sức khỏe Internet
thương mại điện tử
thanh toán di động
thanh toán bên thứ ba
thanh toán NFC
năng lượng thông minh
B2B
B2C
C2B
C2C
O2O
kết nối Internet
thiết bị đeo thông minh
nông nghiệp thông minh
giao thông thông minh
y tế thông minh
dịch vụ khách hàng thông minh
nhà thông minh
cố vấn đầu tư thông minh
văn hóa du lịch thông minh
bảo vệ môi trường thông minh
lưới điện thông minh
tiếp thị thông minh
tiếp thị số
bán lẻ không người
tài chính Internet
tài chính số
Fintech
công nghệ tài chính
tài chính định lượng
ngân hàng mở
Chuyển đổi số
Đổi mới
Công nghiệp 4.0
Tự động hóa
Số hóa
An ninh mạng
điện toán
Sản xuất thông minh
Hệ sinh thái số
Ra quyết định dựa trên dữ liệu
Hoạch định nguồn lực doanh nghiệp
Quản lý quan hệ khách hàng
""".strip().splitlines()


# Chạy lại cấu hình cần chuẩn bị lại input trước khi cài runtime.
INPUTS_READY = False


## 2. Ghi log và dừng khi lệnh gặp lỗi

In [ ]:
def run_logged(command, log_path, *, cwd=None, env=None, echo=True):
    """Lưu log đầy đủ; không bỏ qua mã lỗi của cài đặt hay OCR."""
    command = [str(value) for value in command]
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print(f"Đang chạy: {command[0]} ... | Log: {log_path}", flush=True)
    tail = collections.deque(maxlen=25)
    last_print = time.monotonic()
    with log_path.open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=cwd, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding="utf-8", errors="replace", bufsize=1,
            start_new_session=(os.name == "posix"))
        try:
            for line in process.stdout:
                log.write(line)
                log.flush()
                tail.append(line)
                if echo or time.monotonic() - last_print >= 20:
                    print(line, end="", flush=True)
                    last_print = time.monotonic()
            return_code = process.wait()
        except BaseException:
            if process.poll() is None:
                if os.name == "posix":
                    os.killpg(process.pid, signal.SIGTERM)
                else:
                    process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    if os.name == "posix":
                        os.killpg(process.pid, signal.SIGKILL)
                    else:
                        process.kill()
                    process.wait()
            raise
        finally:
            process.stdout.close()
    if return_code:
        print("".join(tail))
        raise subprocess.CalledProcessError(return_code, command)
    if not echo:
        print("".join(list(tail)[-6:]))
    return log_path

## 3. Tìm PDF đầu vào

Nếu chưa thấy file, thêm dataset PDF ở **Add Input** rồi chạy lại ô này.
Notebook kiểm tra đầu vào trước khi tải môi trường/model. Các PDF có tên trùng
sẽ có hậu tố riêng để không ghi đè kết quả của nhau.

In [ ]:
def select_pdfs(root, requested=(), limit=None):
    root = Path(root).resolve()
    if not root.is_dir():
        raise FileNotFoundError(f"Chưa có thư mục dữ liệu: {root}. Hãy Add Input trên Kaggle.")
    if limit is not None and (type(limit) is not int or limit < 1):
        raise ValueError("FILE_LIMIT phải là số nguyên dương hoặc None.")
    if requested:
        paths = [(Path(name) if Path(name).is_absolute() else root / name).resolve()
                 for name in requested]
        if any(not p.is_file() or p.suffix.lower() != ".pdf" or not p.is_relative_to(root)
               for p in paths):
            raise ValueError("PDF_FILES phải trỏ tới các file PDF tồn tại bên trong INPUT_ROOT.")
    else:
        paths = [p.resolve() for p in root.rglob("*") if p.is_file() and p.suffix.lower() == ".pdf"]
    paths = sorted(set(paths), key=lambda p: str(p).casefold())
    if not paths:
        raise FileNotFoundError(f"Không tìm thấy PDF trong {root}. Hãy Add Input dataset chứa PDF.")
    return paths if limit is None else paths[:limit]


def stage_pdfs(paths, runtime_root):
    identity = [(str(p), p.stat().st_size, p.stat().st_mtime_ns) for p in paths]
    selection_id = hashlib.sha256(json.dumps(identity).encode()).hexdigest()[:12]
    directory = Path(runtime_root) / "selected-inputs" / selection_id
    directory.mkdir(parents=True, exist_ok=True)
    stem_counts = collections.Counter(p.stem.casefold() for p in paths)
    mapping = []
    used_names = set()
    for source in paths:
        suffix = ("-" + hashlib.sha256(str(source).encode()).hexdigest()[:12]
                  if stem_counts[source.stem.casefold()] > 1 else "")
        name = f"{source.stem}{suffix}.pdf"
        if name.casefold() in used_names:
            raise ValueError(f"Tên đầu ra trùng: {name}. Hãy thu hẹp PDF_FILES.")
        used_names.add(name.casefold())
        target = directory / name
        if not target.exists():
            try:
                target.symlink_to(source)
            except OSError:
                shutil.copy2(source, target)
        mapping.append({"source": str(source), "staged": str(target), "document_id": target.stem})
    return directory, mapping


INPUTS_READY = False

if PROFILE not in {"local", "balanced", "accuracy", "fast"}:
    raise ValueError("PROFILE phải là local, balanced, accuracy hoặc fast.")
if DEVICE not in {"cpu", "cuda"} or (PROFILE == "fast" and DEVICE != "cpu"):
    raise ValueError("DEVICE phải là cpu/cuda; PROFILE fast chỉ hỗ trợ cpu.")
if ASSETS not in {"none", "jpeg", "png"}:
    raise ValueError("ASSETS phải là none, jpeg hoặc png.")
if KEYWORDS_FILE is not None and not Path(KEYWORDS_FILE).expanduser().is_file():
    raise FileNotFoundError(f"Không có danh sách keyword: {KEYWORDS_FILE}")
PDF_PATHS = select_pdfs(INPUT_ROOT, PDF_FILES, FILE_LIMIT)
STAGED_INPUT, INPUT_MAPPING = stage_pdfs(PDF_PATHS, RUNTIME_ROOT)
LOG_DIR.mkdir(parents=True, exist_ok=True)
(RESULTS_DIR / "input_manifest.json").write_text(
    json.dumps(INPUT_MAPPING, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Đã chọn {len(PDF_PATHS)} PDF. Mặc định thử {SMOKE_PAGES} trang của file đầu tiên.")
for path in PDF_PATHS[:20]:
    print(path)
if len(PDF_PATHS) > 20:
    print(f"... và {len(PDF_PATHS) - 20} file khác")

INPUTS_READY = True


## 4. Tải mã nguồn và cài Python 3.11 riêng

Cần bật Internet. Lượt cài đầu tiên tải dependency và model; log đầy đủ nằm trong
thư mục `logs`. Chạy lại trong cùng phiên sẽ dùng lại mã nguồn/môi trường đã tạo.
`balanced`/`accuracy` cài thêm môi trường Docling riêng. Lựa chọn CUDA sẽ báo lỗi
nếu không có GPU hoặc GPU/driver không chạy được wheel đã cài.

Cell 3 phải chạy thành công trước cell này. Sau khi khởi động lại kernel hoặc đổi cấu hình ở cell 1, chọn **Run All** từ đầu.


In [ ]:
if not globals().get("INPUTS_READY", False):
    raise RuntimeError("Chưa chuẩn bị xong input. Chạy cell 1 (cấu hình), cell 2 (hàm hỗ trợ), "
                       "cell 3 (chọn PDF) đến khi in số PDF đã chọn, rồi chạy cell 4. "
                       "Sau Restart Session, hãy dùng Run All từ đầu notebook.")

if DEVICE == "cuda":
    if shutil.which("nvidia-smi") is None:
        raise RuntimeError("Chưa có GPU. Bật GPU trong Settings hoặc chọn DEVICE='cpu'.")
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True)
    if gpu.returncode:
        raise RuntimeError("Chưa có GPU. Bật GPU trong Settings hoặc chọn DEVICE='cpu'.")
    print(gpu.stdout)

source_key = hashlib.sha256((REPO_URL + "\n" + REPO_REF).encode()).hexdigest()[:10]
REPO_DIR = RUNTIME_ROOT / f"source-{source_key}"
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        raise RuntimeError(f"Thư mục đã tồn tại nhưng không phải Git checkout: {REPO_DIR}")
    run_logged(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, REPO_DIR],
               LOG_DIR / "clone.log", echo=False)
source_commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"Nguồn đang chạy: {source_commit}")

uv = shutil.which("uv")
if uv is None:
    run_logged([sys.executable, "-m", "pip", "install", "uv==0.12.10"],
               LOG_DIR / "install-uv.log", echo=False)
    uv = shutil.which("uv")
if uv is None:
    raise RuntimeError("Không tìm thấy uv sau khi cài đặt; xem logs/install-uv.log.")

MODELS_DIR = RUNTIME_ROOT / "models"
CACHE_DIR = RUNTIME_ROOT / "cache"
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV.pop("PYTHONPATH", None)
RUNTIME_ENV.update({
    "PATH": str(Path(uv).parent) + os.pathsep + RUNTIME_ENV.get("PATH", ""),
    "PYTHONNOUSERSITE": "1", "PYTHONIOENCODING": "utf-8", "PYTHONUNBUFFERED": "1",
    "UV_CACHE_DIR": str(RUNTIME_ROOT / "uv-cache"),
    "UV_PYTHON_INSTALL_DIR": str(RUNTIME_ROOT / "python"),
    "HF_HOME": str(RUNTIME_ROOT / "huggingface"),
    "OCR_DATA_DIR": str(STAGED_INPUT), "OCR_OUTPUT_DIR": str(BATCH_DIR),
    "OCR_CACHE_DIR": str(CACHE_DIR), "OCR_MODELS_DIR": str(MODELS_DIR),
    "VIETOCR_MODEL_DIR": str(MODELS_DIR / "vietocr"),
    "OCR_DOCLING_PYTHON": str(REPO_DIR / ".venv-docling/bin/python"),
    "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2", "TOKENIZERS_PARALLELISM": "false",
})
OCR_PYTHON = REPO_DIR / ".venv/bin/python"
if not OCR_PYTHON.is_file():
    run_logged([uv, "venv", "--python", "3.11", REPO_DIR / ".venv"],
               LOG_DIR / "python-311.log", env=RUNTIME_ENV, echo=False)
setup_command = ["bash", str(REPO_DIR / "scripts/setup.sh"), "--python", str(OCR_PYTHON),
                 "--device", DEVICE, "--models-dir", str(MODELS_DIR),
                 "--wheel-cache", str(RUNTIME_ROOT / "uv-cache")]
if PROFILE in {"local", "fast"}:
    setup_command.append("--no-docling")
run_logged(setup_command, LOG_DIR / "setup.log", cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)

probe = """
import json,sys,torch
device = sys.argv[1]
if device == 'cuda' and not torch.cuda.is_available():
    raise RuntimeError('CUDA không sẵn sàng trong môi trường OCR riêng')
x = torch.ones((16,16),device=device); value=(x@x)[0,0].item()
print(json.dumps({'python':sys.version,'torch':torch.__version__,'device':str(x.device),
 'gpu':torch.cuda.get_device_name(0) if device=='cuda' else None,'probe':value}))
"""
run_logged([OCR_PYTHON, "-c", probe, DEVICE], LOG_DIR / "device-probe.log",
           cwd=REPO_DIR, env=RUNTIME_ENV)

KEYWORDS_PATH = Path(KEYWORDS_FILE).expanduser().resolve() if KEYWORDS_FILE else REPO_DIR / "config/keywords.txt"
if not KEYWORDS_PATH.is_file():
    raise FileNotFoundError(f"Không có danh sách keyword: {KEYWORDS_PATH}")
HARDWARE_PATH = RUNTIME_ROOT / "kaggle-hardware.yaml"
HARDWARE_PATH.write_text(json.dumps({"hardware": {
    "name": "kaggle", "description": "Kaggle OCR session", "device": DEVICE,
    "workers": 1, "threads_per_worker": 2, "batch_size": 4,
    "paths": {"data_dir": str(STAGED_INPUT), "output_dir": str(BATCH_DIR),
              "cache_dir": str(CACHE_DIR), "models_dir": str(MODELS_DIR)}
}}, indent=2), encoding="utf-8")
(RESULTS_DIR / "run_settings.json").write_text(json.dumps({
    "repository": REPO_URL, "ref": REPO_REF, "source_commit": source_commit,
    "profile": PROFILE, "device": DEVICE, "output_mode": "compact", "assets": ASSETS,
    "keywords_file": str(KEYWORDS_PATH), "smoke_pages": SMOKE_PAGES,
    "full_batch_requested": RUN_FULL_BATCH, "selected_files": len(INPUT_MAPPING),
}, ensure_ascii=False, indent=2), encoding="utf-8")

## 5. Chuẩn bị lệnh và kiểm tra kết quả

In [ ]:
def common_options():
    return ["--profile", PROFILE, "--device", DEVICE, "--hardware", str(HARDWARE_PATH),
            "--assets", ASSETS, "--output-mode", "compact", "--cache-dir", str(CACHE_DIR),
            "--keywords-file", str(KEYWORDS_PATH)]


def smoke_command():
    command = [str(OCR_PYTHON), str(REPO_DIR / "run_ocr.py"), INPUT_MAPPING[0]["staged"],
               "--output", str(SMOKE_DIR), "--pages", SMOKE_PAGES, "--benchmark", *common_options()]
    if RESUME:
        command.append("--resume")
    return command


def batch_command():
    command = [str(OCR_PYTHON), str(REPO_DIR / "run_batch.py"), "--input", str(STAGED_INPUT),
               "--output", str(BATCH_DIR), "--workers", "1", "--threads-per-worker", "2", *common_options()]
    if not RESUME:
        command.append("--force")
    return command


def check_batch_manifest(directory, expected_ids):
    manifest = json.loads((Path(directory) / "batch_manifest.json").read_text(encoding="utf-8"))
    actual_ids = {entry["document_id"] for entry in manifest}
    if actual_ids != set(expected_ids) or len(manifest) != len(expected_ids):
        raise RuntimeError("Danh sách tài liệu trong báo cáo không khớp các PDF đã chọn.")
    failures = [entry for entry in manifest if entry.get("status") not in {"COMPLETED", "CACHED"}]
    if failures:
        details = [{"document_id": e.get("document_id"), "status": e.get("status"),
                    "error": e.get("error")} for e in failures[:10]]
        raise RuntimeError("Có tài liệu xử lý thất bại: " + json.dumps(details, ensure_ascii=False))
    return manifest


def verify_documents(directories, summary_path):
    program = """
import json,sys
from pathlib import Path
from openpyxl import load_workbook
from src.ocr.infrastructure.export.completion import verify_completed
summary=[]
for directory in json.loads(sys.argv[1]):
    root=Path(directory)
    if not verify_completed(root): raise RuntimeError(f'Output chưa hoàn tất/hỏng: {root}')
    validation=json.loads((root/'validation.json').read_text(encoding='utf-8'))
    book=load_workbook(root/'count_keywords.xlsx',read_only=True,data_only=True)
    try:
        rows=list(book.active.iter_rows(min_row=5,values_only=True))
        counts=[(r[1],r[2]) for r in rows[:-1]]
        total=rows[-1][2]
        if sum(count for _,count in counts)!=total: raise RuntimeError('Tổng keyword Excel không khớp')
    finally: book.close()
    summary.append({'document_id':root.name,'validation':validation['status'],
                    'keywords':len(counts),'matches':total,'directory':str(root)})
Path(sys.argv[2]).write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(summary,ensure_ascii=False,indent=2))
"""
    run_logged([OCR_PYTHON, "-c", program, json.dumps([str(p) for p in directories]), summary_path],
               LOG_DIR / (Path(summary_path).stem + ".log"), cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)
    return json.loads(Path(summary_path).read_text(encoding="utf-8"))

## 6. Chạy thử trước

Đọc kết quả trong `ocr_pdf_results/smoke/<DOC_ID>/result.md` và
`count_keywords.xlsx`. `NEEDS_REVIEW` yêu cầu kiểm tra lại nội dung OCR, không có
nghĩa là đã đạt độ chính xác đo bằng ground truth.

In [ ]:
run_logged(smoke_command(), LOG_DIR / "smoke.log", cwd=REPO_DIR, env=RUNTIME_ENV)
SMOKE_SUMMARY = verify_documents([SMOKE_DIR / INPUT_MAPPING[0]["document_id"]],
                                 RESULTS_DIR / "smoke-summary.json")
print("Đã hoàn tất lượt kiểm tra. Xem result.md và count_keywords.xlsx trước khi chạy cả lô.")

## 7. Chạy đủ tài liệu đã chọn

Mặc định `RUN_FULL_BATCH = True`, `FILE_LIMIT = None`, `PDF_FILES = []`: tự tìm
tất cả PDF trong `INPUT_ROOT`, gồm các thư mục con, rồi xử lý đầy đủ mọi trang
và gom vào cùng một ma trận. Có N PDF thì ma trận có N hàng; không giới hạn 50 file
và không thêm hàng giả. Một worker dùng một GPU.

Khi có file thất bại, notebook báo lỗi và chưa tạo ma trận hoàn chỉnh. Có thể
sửa nguyên nhân rồi chạy lại với `RESUME = True` để dùng các kết quả còn hợp lệ.

In [ ]:
if RUN_FULL_BATCH:
    run_logged(batch_command(), LOG_DIR / "batch.log", cwd=REPO_DIR, env=RUNTIME_ENV)
    manifest = check_batch_manifest(BATCH_DIR, [item["document_id"] for item in INPUT_MAPPING])
    BATCH_SUMMARY = verify_documents([BATCH_DIR / item["document_id"] for item in INPUT_MAPPING],
                                     RESULTS_DIR / "batch-summary.json")
    print(f"Đã hoàn tất {len(manifest)} tài liệu.")
else:
    print("Chưa chạy cả lô. Đổi RUN_FULL_BATCH=True để xử lý các PDF đã chọn.")

## 8. Xuất Excel ma trận tổng hợp 88 keyword

Mỗi PDF là một hàng. Cột đầu là tên file; tiếp theo là **88 cột keyword** theo
đúng thứ tự ở `MATRIX_KEYWORDS`. Giữ nguyên hai cột **Internet di động** tại vị trí
keyword 43 và 45; hai cột có cùng số đếm. Không tự loại trùng hoặc đổi tên keyword.
Số 0 là keyword không xuất hiện, chỉ ghi sau khi tài liệu đã xử lý hoàn tất.

Với N PDF trong input, file là `MA_TRAN_DEM_TU_KHOA_<N>_FILES.xlsx`, gồm N hàng
dữ liệu và 88 cột keyword, thêm một cột tên file (tổng 89 cột). Sheet **Thông tin**
ghi số tài liệu thực tế, quy tắc keyword trùng và nguồn PDF. Toàn bộ file được
gom vào cùng một Excel; file Excel được thêm vào ZIP ở bước sau.

Ma trận luôn đếm 88 cột trong `MATRIX_KEYWORDS`; `KEYWORDS_FILE` tùy chỉnh vẫn áp
dụng cho báo cáo `count_keywords.xlsx` riêng từng PDF. Bước này chỉ đọc kết quả
Markdown, không chạy OCR lại. Khi chỉ chạy thử vài trang (`RUN_FULL_BATCH=False`)
thì chưa tạo ma trận cả lô.

In [ ]:
def export_keyword_matrix():
    """Đếm trên output đã hoàn tất, giữ đúng 88 cột kể cả keyword trùng."""
    if len(MATRIX_KEYWORDS) != 88 or any(not isinstance(k, str) or not k.strip() for k in MATRIX_KEYWORDS):
        raise ValueError("MATRIX_KEYWORDS phải chứa đủ 88 từ/cụm từ theo thứ tự yêu cầu.")
    check_batch_manifest(BATCH_DIR, [item["document_id"] for item in INPUT_MAPPING])
    spec_path = RESULTS_DIR / "matrix-input.json"
    spec_path.write_text(json.dumps({"keywords": MATRIX_KEYWORDS, "documents": INPUT_MAPPING,
        "batch_dir": str(BATCH_DIR)}, ensure_ascii=False, indent=2), encoding="utf-8")
    matrix_path = RESULTS_DIR / f"MA_TRAN_DEM_TU_KHOA_{len(INPUT_MAPPING)}_FILES.xlsx"
    program = """
import json,sys,unicodedata
from io import BytesIO
from pathlib import Path
from openpyxl import Workbook,load_workbook
from openpyxl.comments import Comment
from openpyxl.styles import Alignment,Font,PatternFill
from openpyxl.utils import get_column_letter
from src.ocr.infrastructure.export.completion import verify_completed
from src.ocr.infrastructure.postprocessing.keyword_counter import KeywordCounter
from src.ocr.runtime.atomic_io import atomic_write_bytes

spec=json.loads(Path(sys.argv[1]).read_text(encoding='utf-8'))
keywords=spec['keywords']; documents=spec['documents']; root=Path(spec['batch_dir'])
if not documents: raise ValueError('Không có tài liệu để xuất ma trận')
if len(keywords)!=88: raise ValueError('Ma trận yêu cầu đủ 88 cột keyword')
def normalize(value): return unicodedata.normalize('NFC',value.strip()).casefold()
book=Workbook(); sheet=book.active; sheet.title='Ma trận từ khóa'
header_fill=PatternFill('solid',fgColor='1F4E79')
alternate_fill=PatternFill('solid',fgColor='F2F6FC')
headers=['Tệp PDF',*keywords]
for column,label in enumerate(headers,1):
    cell=sheet.cell(1,column,label); cell.data_type='s'
    cell.font=Font(bold=True,color='FFFFFF')
    cell.fill=header_fill
    cell.alignment=Alignment(horizontal='center',vertical='center',wrap_text=True)
seen={}
for index,keyword in enumerate(keywords,1):
    key=normalize(keyword)
    if key in seen:
        sheet.cell(1,index+1).comment=Comment(
            f'Cột keyword thứ {index} lặp cột thứ {seen[key]} theo danh sách gốc; số đếm giống nhau.',
            'OCR PDF')
    else: seen[key]=index
sheet.row_dimensions[1].height=110
sheet.column_dimensions['A'].width=52
for column in range(2,len(headers)+1):
    sheet.column_dimensions[get_column_letter(column)].width=21

source_rows=[]
for row,item in enumerate(documents,2):
    directory=root/item['document_id']
    if not verify_completed(directory):
        raise RuntimeError(f'Tài liệu chưa hoàn tất hoặc output đã hỏng: {directory}')
    markdown=directory/'result.md'
    report=KeywordCounter.count_keywords(markdown.read_text(encoding='utf-8-sig'),keywords,
        item['document_id'],str(markdown))
    counts={normalize(k.keyword):k.count for k in report.keywords}
    file_name=Path(item['staged']).name
    label=sheet.cell(row,1,file_name); label.data_type='s'
    label.alignment=Alignment(vertical='center',wrap_text=True)
    for column,keyword in enumerate(keywords,2):
        cell=sheet.cell(row,column,counts[normalize(keyword)])
        cell.number_format='#,##0'
        cell.alignment=Alignment(horizontal='center',vertical='center')
    if row%2==0:
        for cell in sheet[row]: cell.fill=alternate_fill
    source_rows.append([row-1,file_name,item['source'],item['document_id']])
sheet.freeze_panes='B2'
sheet.auto_filter.ref=f'A1:{get_column_letter(len(headers))}{len(documents)+1}'

info=book.create_sheet('Thông tin')
for row,values in enumerate([
    ['Thuộc tính','Giá trị'],['Số tài liệu',len(documents)],['Số cột keyword',len(keywords)],
    ['Số keyword khác nhau',len(seen)],['Nguồn đếm','result.md của tài liệu đã xử lý đầy đủ'],
    ['Keyword lặp','Giữ nguyên hai cột Internet di động ở vị trí keyword 43 và 45; số đếm giống nhau.'],
    ['Giá trị 0','Keyword không xuất hiện trong Markdown của tài liệu đã hoàn tất.'],
],1):
    for column,value in enumerate(values,1):
        cell=info.cell(row,column,value)
        if isinstance(value,str): cell.data_type='s'
        cell.alignment=Alignment(vertical='top',wrap_text=True)
for column,value in enumerate(['STT','Tệp PDF','Nguồn PDF','Document ID'],1):
    info.cell(9,column,value)
for row,values in enumerate(source_rows,10):
    for column,value in enumerate(values,1):
        cell=info.cell(row,column,value)
        if isinstance(value,str): cell.data_type='s'
for row in (1,9):
    for cell in info[row]:
        cell.font=Font(bold=True,color='FFFFFF'); cell.fill=header_fill
info.column_dimensions['A'].width=27; info.column_dimensions['B'].width=72
info.column_dimensions['C'].width=95; info.column_dimensions['D'].width=55
info.row_dimensions[6].height=34
info.freeze_panes='C10'
try:
    buffer=BytesIO(); book.save(buffer)
    output=Path(sys.argv[2]); atomic_write_bytes(output,buffer.getvalue())
finally: book.close()

verified=load_workbook(output,read_only=True,data_only=True)
try:
    matrix=verified['Ma trận từ khóa']
    if matrix.max_row!=len(documents)+1 or matrix.max_column!=89:
        raise RuntimeError('Kích thước ma trận Excel không khớp')
    if [matrix.cell(1,c).value for c in range(2,90)]!=keywords:
        raise RuntimeError('Thứ tự/cột keyword bị thay đổi')
finally: verified.close()
print(f'Đã xuất {len(documents)} tài liệu x {len(keywords)} cột keyword: {output}')
"""
    run_logged([OCR_PYTHON, "-c", program, spec_path, matrix_path], LOG_DIR / "keyword-matrix.log",
               cwd=REPO_DIR, env=RUNTIME_ENV, echo=False)
    return matrix_path


MATRIX_PATH = None
if RUN_FULL_BATCH:
    MATRIX_PATH = export_keyword_matrix()
    print(f"Excel ma trận tổng hợp: {MATRIX_PATH}")
else:
    print("Chưa xuất ma trận tổng hợp: đặt RUN_FULL_BATCH=True và chạy đủ các tài liệu đã chọn.")

## 9. Đóng gói và tải kết quả

ZIP chứa kết quả OCR, Excel keyword từng file, Excel ma trận tổng hợp 88 keyword, manifest và log; không chứa môi trường hay
model. Tải ZIP qua liên kết bên dưới hoặc phần Output của Kaggle. Dùng **Save Version**
để giữ kết quả của phiên. Nếu một ô trước gặp lỗi, vẫn có thể chạy riêng ô này
để tải log/kết quả đã có; file ZIP khi đó không chứng minh cả lô thành công.

[Tài liệu Kaggle](https://www.kaggle.com/docs/notebooks) ·
[Môi trường Kaggle chính thức](https://github.com/Kaggle/docker-python) ·
[Python riêng bằng uv](https://docs.astral.sh/uv/guides/install-python/)

In [ ]:
def bundle_results(results_dir, archive_path):
    results_dir, archive_path = Path(results_dir), Path(archive_path)
    if not results_dir.is_dir():
        raise FileNotFoundError(f"Chưa có kết quả: {results_dir}")
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    return Path(shutil.make_archive(str(archive_path.with_suffix("")), "zip", root_dir=results_dir))


archive = bundle_results(RESULTS_DIR, ARCHIVE_PATH)
print(f"Kết quả: {RESULTS_DIR}")
print(f"ZIP: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
for summary_file in (RESULTS_DIR / "smoke-summary.json", RESULTS_DIR / "batch-summary.json"):
    if summary_file.is_file():
        print(summary_file.name)
        for entry in json.loads(summary_file.read_text(encoding="utf-8"))[:20]:
            print(f"  {entry['document_id']}: {entry['validation']}, {entry['matches']} lần khớp keyword")
from IPython.display import FileLink, display

def download_link(path):
    try:
        link = os.path.relpath(path, Path.cwd())
    except ValueError:  # Local Windows verification may use a different drive.
        link = Path(path).as_posix()
    return FileLink(link)

matrix_path = globals().get("MATRIX_PATH")
if matrix_path is not None and Path(matrix_path).is_file():
    display(download_link(matrix_path))
display(download_link(archive))